# Build Databases
- using duckdb to build the database

In [1]:
import duckdb

In [2]:
#build the database grid with duckdb
con = duckdb.connect("../data/grid.duckdb") 
#put in data folder, this will create a new database if it doesn't exist, or connect to an existing one

In [4]:
#query the csv data
con.sql("SELECT * FROM '../data/raw/eia_bpat_hourly.csv'")

┌──────────────────────────┬────────────┬─────────┬───────────┐
│        period_utc        │ respondent │  type   │ value_mwh │
│ timestamp with time zone │  varchar   │ varchar │  double   │
├──────────────────────────┼────────────┼─────────┼───────────┤
│ 2020-12-31 16:00:00-08   │ BPAT       │ D       │    6839.0 │
│ 2020-12-31 16:00:00-08   │ BPAT       │ DF      │    7125.0 │
│ 2020-12-31 17:00:00-08   │ BPAT       │ D       │    7233.0 │
│ 2020-12-31 17:00:00-08   │ BPAT       │ DF      │    7445.0 │
│ 2020-12-31 18:00:00-08   │ BPAT       │ D       │    7434.0 │
│ 2020-12-31 18:00:00-08   │ BPAT       │ DF      │    7696.0 │
│ 2020-12-31 19:00:00-08   │ BPAT       │ D       │    7263.0 │
│ 2020-12-31 19:00:00-08   │ BPAT       │ DF      │    7527.0 │
│ 2020-12-31 20:00:00-08   │ BPAT       │ D       │    6999.0 │
│ 2020-12-31 20:00:00-08   │ BPAT       │ DF      │    7208.0 │
│           ·              │  ·         │ ·       │       ·   │
│           ·              │  ·         

In [5]:
#as a pandas dataframe
con.sql("SELECT * FROM '../data/raw/eia_bpat_hourly.csv'").df()

,period_utc,respondent,type,value_mwh
0,2020-12-31 16:00:00-08:00,BPAT,D,6839.0
1,2020-12-31 16:00:00-08:00,BPAT,DF,7125.0
2,2020-12-31 17:00:00-08:00,BPAT,D,7233.0
3,2020-12-31 17:00:00-08:00,BPAT,DF,7445.0
4,2020-12-31 18:00:00-08:00,BPAT,D,7434.0
...,...,...,...,...
99235,2026-08-31 14:00:00-07:00,BPAT,DF,7452.0
99236,2026-08-31 15:00:00-07:00,BPAT,D,7704.0
99237,2026-08-31 15:00:00-07:00,BPAT,DF,7601.0
99238,2026-08-31 16:00:00-07:00,BPAT,D,7884.0


'D' is the demand found and 'DF' is the day-ahead demand forecast predicted

In [ ]:
#now actually turn the data into a table with CREATE TABLE statement

#EIA data
con.sql("""
CREATE OR REPLACE TABLE eia_raw AS
SELECT * FROM read_csv_auto('../data/raw/eia_bpat_hourly.csv')
""")
#selects all the data from the csv and creates a new table called eia_raw in the duckdb database

In [8]:
#open-meteo data
con.sql("""
CREATE OR REPLACE TABLE weather_raw AS
SELECT * FROM read_csv_auto('../data/raw/weather_hourly.csv')
""")

In [9]:
#Now look at the tables in the database
con.sql("SHOW TABLES").df()

,name
0,eia_raw
1,weather_raw


In [10]:
con.sql("DESCRIBE eia_raw")

┌─────────────┬──────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │       column_type        │  null   │   key   │ default │  extra  │
│   varchar   │         varchar          │ varchar │ varchar │ varchar │ varchar │
├─────────────┼──────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ period_utc  │ TIMESTAMP WITH TIME ZONE │ YES     │ NULL    │ NULL    │ NULL    │
│ respondent  │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ type        │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ value_mwh   │ DOUBLE                   │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴──────────────────────────┴─────────┴─────────┴─────────┴─────────┘

In [11]:
con.sql("DESCRIBE weather_raw")

┌──────────────────────┬──────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name      │       column_type        │  null   │   key   │ default │  extra  │
│       varchar        │         varchar          │ varchar │ varchar │ varchar │ varchar │
├──────────────────────┼──────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ location             │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ period_utc           │ TIMESTAMP WITH TIME ZONE │ YES     │ NULL    │ NULL    │ NULL    │
│ temperature_2m       │ DOUBLE                   │ YES     │ NULL    │ NULL    │ NULL    │
│ apparent_temperature │ DOUBLE                   │ YES     │ NULL    │ NULL    │ NULL    │
│ relative_humidity_2m │ BIGINT                   │ YES     │ NULL    │ NULL    │ NULL    │
│ wind_speed_10m       │ DOUBLE                   │ YES     │ NULL    │ NULL    │ NULL    │
│ cloud_cover          │ BIGINT                   │ YES     │ NULL    │ NULL    

Make some first queries to explore the databases some more. 

In [28]:
#rows for EIA
con.sql("SELECT COUNT(*) AS n_rows FROM eia_raw")

┌────────┐
│ n_rows │
│ int64  │
├────────┤
│  99240 │
└────────┘

In [30]:
#rows by type
con.sql("SELECT type, COUNT(*) AS n_rows " \
"FROM eia_raw " \
"GROUP BY type;")

┌─────────┬────────┐
│  type   │ n_rows │
│ varchar │ int64  │
├─────────┼────────┤
│ DF      │  49584 │
│ D       │  49656 │
└─────────┴────────┘

Different number of rows per type, difference is 72. So missing 72 hours from 'DF'

In [31]:
#min and max time values
con.sql("SELECT MIN(period_utc) as min_time, " \
        "MAX(period_utc) AS max_time "
"FROM eia_raw;")

┌──────────────────────────┬──────────────────────────┐
│         min_time         │         max_time         │
│ timestamp with time zone │ timestamp with time zone │
├──────────────────────────┼──────────────────────────┤
│ 2020-12-31 16:00:00-08   │ 2026-08-31 16:00:00-07   │
└──────────────────────────┴──────────────────────────┘

In [32]:
#weather data
#how many rows per location
con.sql("SELECT location, COUNT(*) as n_rows " \
"FROM weather_raw " \
"GROUP BY location")

┌────────────────┬────────┐
│    location    │ n_rows │
│    varchar     │ int64  │
├────────────────┼────────┤
│ Portland       │  49656 │
│ Eugene         │  49656 │
│ Vancouver_wash │  49656 │
│ Spokane        │  49656 │
└────────────────┴────────┘

In [35]:
#any null/0 
con.sql("SELECT type, "
        "COUNT(*) AS N_bad, " \
        "SUM(CASE WHEN value_mwh IS NULL THEN 1 ELSE 0 END) AS n_null,"
        "SUM(CASE WHEN value_mwh = 0 THEN 1 ELSE 0 END) AS n_zero,"
"FROM eia_raw " \
"WHERE value_mwh is NULL or value_mwh=0"
"GROUP BY type")

┌─────────┬───────┬────────┬────────┐
│  type   │ N_bad │ n_null │ n_zero │
│ varchar │ int64 │ int128 │ int128 │
├─────────┼───────┼────────┼────────┤
│ D       │    49 │     49 │      0 │
└─────────┴───────┴────────┴────────┘

Nothing bad in 'DF' but have 49 bad counts in 'D' and all have them are 'NULL' cases. Could be part of the difference in between 'D' and 'DF'

In [37]:
#close the connection
con.close()